[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rpasquini/econometrics-book-notebooks/blob/main/es/ch1_regresion_simple/simple_linear_regression_notebook.ipynb)

# Cuaderno de código: Regresión Lineal Simple

Este cuaderno acompaña la sección [El Modelo de Regresión Lineal Simple](simple_linear_regression.md). El dashboard interactivo de esa sección sirve para *sentir* el problema moviendo la recta a mano; este cuaderno muestra cómo se resuelve el mismo problema en un **pipeline real de análisis**: cargar datos, explorarlos, ajustar el modelo con una librería estándar, e interpretar y diagnosticar el resultado.

Vamos a trabajar con dos conjuntos de datos:

1. **Las 140 campañas publicitarias** de la sección — el mismo dato que genera el dashboard, congelado en un archivo `.csv` para que el análisis sea reproducible.
2. **Los datos de gasto en alimentos de Engel (1857)**, un ejemplo clásico de la literatura econométrica, para ver el mismo pipeline aplicado a datos reales publicados en un paper.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
import statsmodels.formula.api as smf


## Parte 1 — Las 140 campañas publicitarias

`budget_usd_k` es el presupuesto invertido en publicidad (miles de USD); `revenue_usd_k` son los ingresos generados por la campaña (miles de USD). Es el mismo dato que ves en el dashboard de la sección, generado con la misma semilla aleatoria.

In [ ]:
df = pd.read_csv('https://raw.githubusercontent.com/rpasquini/econometrics-book-notebooks/main/es/ch1_regresion_simple/data/advertising_campaigns.csv')
df.describe()


In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(df['budget_usd_k'], df['revenue_usd_k'], alpha=0.6)
ax.set_xlabel('Presupuesto publicitario (miles de USD)')
ax.set_ylabel('Ingresos (miles de USD)')
ax.set_title('Campañas publicitarias: presupuesto vs. ingresos')
plt.show()


### Ajuste del modelo

Usamos `statsmodels.formula.api.ols`, la forma estándar de ajustar una regresión lineal en Python cuando los datos están en un `DataFrame`.

In [ ]:
model_ads = smf.ols('revenue_usd_k ~ budget_usd_k', data=df).fit()
model_ads.summary()


**Interpretación.** $\hat\beta_0$ es el ingreso promedio estimado cuando el presupuesto publicitario es cero; $\hat\beta_1$ es el ingreso adicional promedio (en miles de USD) por cada mil dólares adicionales de presupuesto. Compará estos valores con los parámetros verdaderos que se usaron para simular los datos ($\beta_0=10$, $\beta_1=2$) — la estimación es cercana, pero no exacta, porque cada muestra trae su propio ruido.

### Diagnóstico: residuos vs. valores ajustados

Un chequeo rápido de especificación: si el modelo lineal es adecuado, los residuos no deberían mostrar un patrón sistemático contra los valores ajustados.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(model_ads.fittedvalues, model_ads.resid, alpha=0.6)
ax.axhline(0, color='black', linewidth=1, linestyle='--')
ax.set_xlabel('Valores ajustados')
ax.set_ylabel('Residuos')
ax.set_title('Residuos vs. valores ajustados')
plt.show()


## Parte 2 — Un ejemplo de la literatura: los datos de Engel (1857)

Ernst Engel estudió la relación entre el ingreso de un hogar y su gasto en alimentos, dando origen a la conocida "Ley de Engel". Este conjunto de datos —235 hogares belgas de clase trabajadora— fue popularizado por Koenker y Bassett (1982) en su paper sobre regresión de cuantiles, y viene incluido directamente en `statsmodels`, así que no hace falta descargar nada.

> Koenker, R. y Bassett, G. (1982). *Robust Tests of Heteroscedasticity based on Regression Quantiles.* Econometrica, 50(1), 43–61.

Repetimos exactamente el mismo pipeline: cargar, explorar, ajustar, interpretar.

In [ ]:
engel = sm.datasets.engel.load_pandas().data
engel.describe()


In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(engel['income'], engel['foodexp'], alpha=0.5)
ax.set_xlabel('Ingreso del hogar')
ax.set_ylabel('Gasto en alimentos')
ax.set_title('Datos de Engel (1857): ingreso vs. gasto en alimentos')
plt.show()


In [ ]:
model_engel = smf.ols('foodexp ~ income', data=engel).fit()
model_engel.summary()


**Comparación.** El pipeline es idéntico al de la Parte 1 — mismo código, mismos tres pasos (`ols`, `.fit()`, `.summary()`) — pero acá los datos son observacionales, no simulados: no conocemos los parámetros verdaderos, y el gráfico de dispersión ya deja ver que la dispersión de los residuos crece con el ingreso (heterocedasticidad), algo que vamos a retomar formalmente en el capítulo de Estructura del Error.

## Para explorar por tu cuenta

- Cambiá la especificación del modelo de Engel a log-log (`np.log(foodexp) ~ np.log(income)`) y compará el ajuste.
- Calculá un intervalo de confianza para $\hat\beta_1$ en el modelo de campañas publicitarias con `model_ads.conf_int()`.
- Probá con una submuestra más chica (por ejemplo, las primeras 20 campañas) y observá cómo cambia la precisión de la estimación.